In [6]:
!pip install kaggle
!kaggle competitions download -c asr-2026-spoken-numbers-recognition-challenge

asr-2026-spoken-numbers-recognition-challenge.zip: Skipping, found more recently modified local copy (use --force to force download)


In [8]:
!pip install torch torchaudio librosa numpy pandas tqdm scipy num2words jiwer soundfile pydub

In [ ]:
import zipfile
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torchaudio
import librosa
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from jiwer import cer

In [9]:
with zipfile.ZipFile('asr-2026-spoken-numbers-recognition-challenge.zip', 'r') as zip_ref:
    zip_ref.extractall('data')

In [25]:
DATA_DIR = "/kaggle/working/data"
TRAIN_CSV = os.path.join(DATA_DIR, "train.csv")
DEV_CSV = os.path.join(DATA_DIR, "dev.csv")
TEST_CSV = os.path.join(DATA_DIR, "test.csv")
OUTPUT_FILE = "submission.csv"

# ------------------------ НОРМАЛИЗАЦИЯ ЧИСЕЛ ------------------------
units = ['', 'один', 'два', 'три', 'четыре', 'пять', 'шесть', 'семь', 'восемь', 'девять']
units_female = ['', 'одна', 'две', 'три', 'четыре', 'пять', 'шесть', 'семь', 'восемь', 'девять']
tens = ['', '', 'двадцать', 'тридцать', 'сорок', 'пятьдесят', 'шестьдесят', 'семьдесят', 'восемьдесят', 'девяносто']
teens = ['десять', 'одиннадцать', 'двенадцать', 'тринадцать', 'четырнадцать', 'пятнадцать', 'шестнадцать', 'семнадцать', 'восемнадцать', 'девятнадцать']
hundreds = ['', 'сто', 'двести', 'триста', 'четыреста', 'пятьсот', 'шестьсот', 'семьсот', 'восемьсот', 'девятьсот']

def int_to_russian_words(num: int) -> str:
    if num == 0:
        return "ноль"
    thousands = num // 1000
    remainder = num % 1000
    parts = []
    if thousands > 0:
        if thousands == 1:
            parts.append("одна тысяча")
        elif 2 <= thousands <= 4:
            parts.append(units_female[thousands] + " тысячи")
        else:
            parts.append(int_to_words_under_1000(thousands, is_thousands=True) + " тысяч")
    if remainder > 0:
        if remainder < 10 and thousands > 0 and remainder > 0:
            parts.append(units[remainder])
        else:
            parts.append(int_to_words_under_1000(remainder, is_thousands=False))
    return " ".join(parts)

def int_to_words_under_1000(n: int, is_thousands: bool = False) -> str:
    if n == 0:
        return ""
    h = n // 100
    t = (n % 100) // 10
    u = n % 10
    result = []
    if h > 0:
        result.append(hundreds[h])
    if t == 1:
        result.append(teens[u])
    else:
        if t > 1:
            result.append(tens[t])
        if u > 0:
            if is_thousands and u == 1 and t != 1:
                result.append(units_female[u])
            else:
                result.append(units[u])
    return " ".join(result)

ones_dict = {
    'один': 1, 'одна': 1, 'два': 2, 'две': 2, 'три': 3, 'четыре': 4,
    'пять': 5, 'шесть': 6, 'семь': 7, 'восемь': 8, 'девять': 9
}
teens_dict = {
    'десять': 10, 'одиннадцать': 11, 'двенадцать': 12, 'тринадцать': 13,
    'четырнадцать': 14, 'пятнадцать': 15, 'шестнадцать': 16, 'семнадцать': 17,
    'восемнадцать': 18, 'девятнадцать': 19
}
tens_dict = {
    'двадцать': 20, 'тридцать': 30, 'сорок': 40, 'пятьдесят': 50,
    'шестьдесят': 60, 'семьдесят': 70, 'восемьдесят': 80, 'девяносто': 90
}
hundreds_dict = {
    'сто': 100, 'двести': 200, 'триста': 300, 'четыреста': 400,
    'пятьсот': 500, 'шестьсот': 600, 'семьсот': 700, 'восемьсот': 800,
    'девятьсот': 900
}

def parse_under_1000(words):
    num = 0
    i = 0
    n = len(words)
    while i < n:
        w = words[i]
        if w in hundreds_dict:
            num += hundreds_dict[w]
            i += 1
        elif w in tens_dict:
            num += tens_dict[w]
            i += 1
            if i < n and words[i] in ones_dict:
                num += ones_dict[words[i]]
                i += 1
        elif w in teens_dict:
            num += teens_dict[w]
            i += 1
        elif w in ones_dict:
            num += ones_dict[w]
            i += 1
        else:
            i += 1
    return num

def words_to_int(phrase: str) -> int:
    phrase = phrase.lower().strip()
    words = phrase.split()
    thousand_idx = -1
    for i, w in enumerate(words):
        if w in ('тысяча', 'тысячи', 'тысяч'):
            thousand_idx = i
            break
    if thousand_idx == -1:
        return parse_under_1000(words)
    thousand_words = words[:thousand_idx]
    rest_words = words[thousand_idx+1:]
    if thousand_idx == 0:
        thousands = 1
    else:
        thousands = parse_under_1000(thousand_words)
    remainder = parse_under_1000(rest_words) if rest_words else 0
    return thousands * 1000 + remainder

TARGET_SR = 16000
N_MELS = 80
N_FFT = 400
HOP_LEN = 160
WIN_LEN = 400

def load_audio(file_path, target_sr=TARGET_SR):
    try:
        waveform, orig_sr = torchaudio.load(file_path)
        if orig_sr != target_sr:
            resampler = torchaudio.transforms.Resample(orig_sr, target_sr)
            waveform = resampler(waveform)
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)
        return waveform.squeeze().numpy()
    except Exception as e:
        print(f"Ошибка загрузки {file_path}: {e}")
        return None

def get_mel_spectrogram(audio, sr=TARGET_SR):
    mel_spec = torchaudio.transforms.MelSpectrogram(
        sample_rate=sr, n_fft=N_FFT, hop_length=HOP_LEN, win_length=WIN_LEN,
        n_mels=N_MELS, power=2.0
    )(torch.from_numpy(audio).unsqueeze(0))
    log_mel = torch.log(mel_spec + 1e-6)
    return log_mel.squeeze(0).t().numpy()

class NoiseAugmentation:
    def __init__(self, noise_files=None, snr_range=(5, 20)):
        self.noise_files = noise_files or []
        self.snr_range = snr_range
    def apply(self, audio, sr):
        if len(self.noise_files) == 0:
            return audio
        noise_path = random.choice(self.noise_files)
        noise = load_audio(noise_path, sr)
        if noise is None or len(noise) < len(audio):
            noise = np.tile(noise, int(np.ceil(len(audio)/len(noise))))[:len(audio)]
        snr_db = random.uniform(*self.snr_range)
        snr_linear = 10**(snr_db/10)
        noise_power = np.mean(noise**2)
        signal_power = np.mean(audio**2)
        scale = np.sqrt(signal_power / (noise_power * snr_linear + 1e-8))
        augmented = audio + scale * noise[:len(audio)]
        return augmented

class SpeedPerturbation:
    def __init__(self, factors=[0.85, 0.95, 1.0, 1.05, 1.15]):
        self.factors = factors
    def apply(self, audio, sr):
        factor = random.choice(self.factors)
        if factor == 1.0:
            return audio
        new_sr = int(sr * factor)
        audio_tensor = torch.from_numpy(audio).float()
        resampled = torchaudio.functional.resample(audio_tensor, sr, new_sr)
        resampled_back = torchaudio.functional.resample(resampled, new_sr, TARGET_SR)
        return resampled_back.numpy()

class VolumePerturbation:
    def __init__(self, gain_range=(-20, 20)):
        self.gain_range = gain_range
    def apply(self, audio):
        gain_db = random.uniform(*self.gain_range)
        gain_linear = 10**(gain_db/20)
        return audio * gain_linear

class PitchShift:   # НОВО! изменение высоты тона
    def __init__(self, n_steps_range=(-3, 3), sample_rate=TARGET_SR):
        self.n_steps_range = n_steps_range
        self.sample_rate = sample_rate
    def apply(self, audio):
        n_steps = random.randint(*self.n_steps_range)
        if n_steps == 0:
            return audio
        return librosa.effects.pitch_shift(audio, sr=self.sample_rate, n_steps=n_steps)

class AugmentationPipeline:
    def __init__(self, noise_aug=None, speed_aug=None, volume_aug=None, pitch_aug=None):
        self.noise_aug = noise_aug
        self.speed_aug = speed_aug
        self.volume_aug = volume_aug
        self.pitch_aug = pitch_aug
    def apply(self, audio, sr):
        if self.pitch_aug:
            audio = self.pitch_aug.apply(audio)
        if self.speed_aug:
            audio = self.speed_aug.apply(audio, sr)
        if self.volume_aug:
            audio = self.volume_aug.apply(audio)
        if self.noise_aug:
            audio = self.noise_aug.apply(audio, sr)
        return audio

def spec_augment(mel_spec, time_mask=15, freq_mask=12):
    mel = mel_spec.copy()
    t, f = mel.shape
    for _ in range(3):
        t0 = random.randint(0, max(1, t - time_mask))
        mel[t0:t0+time_mask, :] = 0
    for _ in range(3):
        f0 = random.randint(0, max(1, f - freq_mask))
        mel[:, f0:f0+freq_mask] = 0
    return mel

class GlobalMelNormalizer:
    def __init__(self):
        self.mean = None
        self.std = None
    def fit(self, dataset):
        all_mels = []
        for i in tqdm(range(len(dataset)), desc="Global stats"):
            mel, _, _, _ = dataset[i]
            all_mels.append(mel.numpy().flatten())
        all_mels = np.concatenate(all_mels)
        self.mean = np.mean(all_mels)
        self.std = np.std(all_mels)
        print(f"Global mean: {self.mean:.4f}, std: {self.std:.4f}")
    def normalize(self, mel):
        return (mel - self.mean) / (self.std + 1e-6)

class RussianNumbersDataset(Dataset):
    def __init__(self, csv_file, audio_root, vocab, aug_pipeline=None, augment=True, global_norm=None):
        self.data = pd.read_csv(csv_file)
        self.audio_root = audio_root
        self.vocab = vocab
        self.char2idx = {ch: i for i, ch in enumerate(vocab)}
        self.aug_pipeline = aug_pipeline
        self.augment = augment
        self.global_norm = global_norm

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        full_path = os.path.join(self.audio_root, row['filename'])
        audio = load_audio(full_path)
        if audio is None:
            audio = np.zeros(TARGET_SR)
        if self.augment and self.aug_pipeline:
            audio = self.aug_pipeline.apply(audio, TARGET_SR)
        mel = get_mel_spectrogram(audio, TARGET_SR)
        if self.augment:
            mel = spec_augment(mel)
        if self.global_norm is not None:
            mel = self.global_norm.normalize(mel)
        else:
            mel = (mel - mel.mean()) / (mel.std() + 1e-6)
        if self.augment and random.random() < 0.2:
            freq_mask = random.randint(1, N_MELS//10)
            f0 = random.randint(0, N_MELS - freq_mask)
            mel[:, f0:f0+freq_mask] = 0
        text = row['transcription']
        text_normalized = int_to_russian_words(int(text))
        target_seq = [self.char2idx[ch] for ch in text_normalized if ch in self.char2idx]
        target_tensor = torch.LongTensor(target_seq)
        mel_tensor = torch.FloatTensor(mel)
        return mel_tensor, target_tensor, len(mel_tensor), len(target_tensor)

def collate_fn(batch):
    mels, targets, mel_lengths, target_lengths = zip(*batch)
    max_mel_len = max(mel_lengths)
    max_target_len = max(target_lengths)
    padded_mels = torch.zeros(len(batch), max_mel_len, N_MELS)
    padded_targets = torch.zeros(len(batch), max_target_len, dtype=torch.long)
    for i, (mel, tgt, ml, tl) in enumerate(zip(mels, targets, mel_lengths, target_lengths)):
        padded_mels[i, :ml, :] = mel
        padded_targets[i, :tl] = tgt
    return padded_mels, padded_targets, torch.IntTensor(mel_lengths), torch.IntTensor(target_lengths)

class SimpleASRModel(nn.Module):
    def __init__(self, n_mels=80, hidden_dim=128, num_layers=2, num_classes=41, dropout=0.4):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, 3, stride=(2,2), padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Dropout2d(0.2),
            nn.Conv2d(32, 64, 3, stride=(2,2), padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Dropout2d(0.2),
            nn.Conv2d(64, 128, 3, stride=(1,2), padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
        )
        self.conv_output_size = 128 * (n_mels // 8)
        self.dropout_in = nn.Dropout(dropout)
        self.gru = nn.GRU(self.conv_output_size, hidden_dim, num_layers,
                          bidirectional=True, dropout=dropout, batch_first=True)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)
        self.dropout_out = nn.Dropout(dropout)

    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.conv(x)
        x = x.permute(0, 2, 1, 3).contiguous()
        B, T, C, F = x.shape
        x = x.view(B, T, C * F)
        x = self.dropout_in(x)
        x, _ = self.gru(x)
        x = self.dropout_out(x)
        return self.fc(x)

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for mels, targets, mel_lens, target_lens in tqdm(loader, desc="Train"):
        mels, targets = mels.to(device), targets.to(device)
        mel_lens, target_lens = mel_lens.to(device), target_lens.to(device)
        optimizer.zero_grad()
        logits = model(mels)
        log_probs = torch.log_softmax(logits, dim=-1).permute(1, 0, 2)
        input_lengths = (mel_lens // 4).clamp(min=1)
        loss = criterion(log_probs, targets, input_lengths, target_lens)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate(model, loader, device, vocab):
    model.eval()
    total_cer = 0
    total_samples = 0
    with torch.no_grad():
        for mels, targets, mel_lens, target_lens in tqdm(loader, desc="Val"):
            mels = mels.to(device)
            logits = model(mels)
            pred_ids = torch.argmax(logits, dim=-1)
            for i in range(len(pred_ids)):
                pred_chars = []
                prev = -1
                for idx in pred_ids[i]:
                    if idx != prev and idx != 0:
                        pred_chars.append(vocab[idx])
                    prev = idx
                pred_text = ''.join(pred_chars)
                true_text = ''.join([vocab[idx] for idx in targets[i] if idx != 0])
                total_cer += cer(true_text, pred_text)
                total_samples += 1
    return total_cer / total_samples

def predict_test_set(model, test_csv, audio_root, device, vocab, global_norm=None):
    test_df = pd.read_csv(test_csv)
    predictions = []
    model.eval()
    with torch.no_grad():
        for idx, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Inference"):
            audio_path = os.path.join(audio_root, row['filename'])
            audio = load_audio(audio_path)
            if audio is None:
                pred_int = 0
            else:
                mel = get_mel_spectrogram(audio, TARGET_SR)
                if global_norm is not None:
                    mel = global_norm.normalize(mel)
                else:
                    mel = (mel - mel.mean()) / (mel.std() + 1e-6)
                mel_tensor = torch.FloatTensor(mel).unsqueeze(0).to(device)
                logits = model(mel_tensor)
                pred_ids = torch.argmax(logits, dim=-1)[0]
                pred_chars = []
                prev = -1
                for idx_char in pred_ids:
                    if idx_char != prev and idx_char != 0:
                        pred_chars.append(vocab[idx_char])
                    prev = idx_char
                pred_text = ''.join(pred_chars)
                try:
                    pred_int = words_to_int(pred_text)
                except:
                    pred_int = 0
            predictions.append(pred_int)
    test_df['transcription'] = predictions
    test_df[['filename', 'transcription']].to_csv(OUTPUT_FILE, index=False)
    print(f"Сохранён {OUTPUT_FILE}")

def main():
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Используем устройство: {device}")

chars = " абвгдеёжзийклмнопрстуфхцчшщъыьэюя"
vocab = ['<blank>'] + list(chars)
num_classes = len(vocab)
noise_files = []
noise_aug = NoiseAugmentation(noise_files) if noise_files else None
speed_aug = SpeedPerturbation(factors=[0.85, 0.95, 1.0, 1.05, 1.15])
volume_aug = VolumePerturbation(gain_range=(-20, 20))
pitch_aug = PitchShift(n_steps_range=(-3, 3))
aug_pipeline = AugmentationPipeline(
        noise_aug=noise_aug,
        speed_aug=speed_aug,
        volume_aug=volume_aug,
        pitch_aug=pitch_aug
    )

temp_dataset = RussianNumbersDataset(TRAIN_CSV, DATA_DIR, vocab, augment=False, global_norm=None)
global_norm = GlobalMelNormalizer()
global_norm.fit(temp_dataset)

train_dataset = RussianNumbersDataset(TRAIN_CSV, DATA_DIR, vocab, aug_pipeline, augment=True, global_norm=global_norm)
dev_dataset = RussianNumbersDataset(DEV_CSV, DATA_DIR, vocab, None, augment=False, global_norm=global_norm)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn, num_workers=2, drop_last=True)
dev_loader = DataLoader(dev_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn, num_workers=2)

model = SimpleASRModel(n_mels=N_MELS, hidden_dim=128, num_layers=2, num_classes=num_classes, dropout=0.4)
print(f"Количество параметров: {count_parameters(model)}")
assert count_parameters(model) < 5_000_000

model.to(device)
optimizer = optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-3)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30, eta_min=1e-5)
criterion = nn.CTCLoss(blank=0, zero_infinity=True)

best_cer = float('inf')
patience = 10
no_improve = 0
epochs = 30

for epoch in range(1, epochs+1):
    train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
    val_cer = evaluate(model, dev_loader, device, vocab)
    scheduler.step()
    print(f"Epoch {epoch:2d}: loss={train_loss:.4f}, val_CER={val_cer:.4f}")
    if val_cer < best_cer:
        best_cer = val_cer
        torch.save(model.state_dict(), "best_model.pth")
        print(" -> сохранена лучшая модель")
        no_improve = 0
    else:
        no_improve += 1
        if no_improve >= patience:
            print("Early stopping")
            break

print("Обучение завершено. Запуск инференса...")
model.load_state_dict(torch.load("best_model.pth", map_location=device))
predict_test_set(model, TEST_CSV, DATA_DIR, device, vocab, global_norm=global_norm)

Используем устройство: cuda


Global stats: 100%|██████████| 12553/12553 [03:15<00:00, 64.30it/s]


Global mean: -0.0000, std: 1.0000
Количество параметров: 1481443


Train:   0%|          | 0/392 [00:00<?, ?it/s]Exception ignored in: Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7e6346787880><function _MultiProcessingDataLoaderIter.__del__ at 0x7e6346787880>

Traceback (most recent call last):
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
      File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()self._shutdown_workers()

  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
      File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
if w.is_alive():    
if w.is_alive(): 
            ^^ ^^^^^^^^^^^^^^^^^^^^^^

  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
  File "/usr/lib/python3.12/multiprocessing/process.p

Epoch  1: loss=2.3717, val_CER=0.5679
 -> сохранена лучшая модель


...

Epoch 24: loss=0.0460, val_CER=0.0395


Val: 100%|██████████| 71/71 [00:20<00:00,  3.46it/s]


Epoch 25: loss=0.0456, val_CER=0.0391
Early stopping
Обучение завершено. Запуск инференса...


Inference: 100%|██████████| 2582/2582 [00:45<00:00, 57.16it/s]


Сохранён submission.csv
